# Ingestion

In [2]:
import numpy as np
from openai import OpenAI
from sqlitesearch import TextSearchIndex

print("Kernel is working ✅")

Kernel is working ✅


In [3]:
import ingest

In [4]:
# test loading data
documents = ingest.load_jobs_data()

Loaded 1 job listings
Loaded 2 job listings
Loaded 3 job listings
Loaded 4 job listings
Loaded 5 job listings
Loaded 6 job listings
Loaded 7 job listings
Loaded 8 job listings
Loaded 9 job listings
Loaded 10 job listings
Loaded 11 job listings
Loaded 12 job listings
Loaded 13 job listings
Loaded 14 job listings
Loaded 15 job listings
Loaded 16 job listings
Loaded 17 job listings
Loaded 18 job listings
Loaded 19 job listings
Loaded 20 job listings
Loaded 21 job listings
Loaded 22 job listings
Loaded 23 job listings
Loaded 24 job listings
Loaded 25 job listings
Loaded 26 job listings
Loaded 27 job listings
Loaded 28 job listings
Loaded 29 job listings
Loaded 30 job listings
Loaded 31 job listings
Loaded 32 job listings
Loaded 33 job listings
Loaded 34 job listings
Loaded 35 job listings
Loaded 36 job listings
Loaded 37 job listings
Loaded 38 job listings
Loaded 39 job listings
Loaded 40 job listings
Loaded 41 job listings
Loaded 42 job listings
Loaded 43 job listings
Loaded 44 job listin

In [5]:
# check it loaded
len(documents)

4255

In [6]:
# see first document
documents[9]

{'Title': 'Strategy Analyst (Bangkok Based, Relocation Provided)',
 'Job_Description': 'About AgodaAt Agoda, we bridge the world through travel. Our story began in 2005, when two lifelong friends and entrepreneurs, driven by their passion for travel, launched Agoda to make it easier for everyone to explore the world.Today, we are part of Booking Holdings [NASDAQ: BKNG], with a diverse team of over 7,000 people from 90 countries, working together in offices around the globe. Every day, we connect people to destinations and experiences, with our great deals across our millions of hotels and holiday properties, flights, and experiences worldwide.No two days are the same at Agoda. Data and technology are at the heart of our culture, fueling our curiosity and innovation. If you’re ready to begin your best journey and help build travel for the world, join us.Get To Know The Team:The Performance Marketing Team of Agoda is a world leader in online marketing. This department is highly data-driv

In [7]:
# test keyword index
ingest.build_keyword_index(documents[:5])

Keyword index saved to jobs.db


In [8]:
# test vector index with small sample
ingest.build_vector_index(documents[:3])

Embedding batch 1


In [9]:
# See the keyword index database
import sqlite3

conn = sqlite3.connect(ingest.DB_PATH)

tables = conn.execute("""
SELECT name 
FROM sqlite_master 
WHERE type='table'
""").fetchall()

tables

[('docs',),
 ('sqlite_sequence',),
 ('docs_fts',),
 ('docs_fts_data',),
 ('docs_fts_idx',),
 ('docs_fts_content',),
 ('docs_fts_docsize',),
 ('docs_fts_config',)]

In [10]:
# Then see what is inside each table:
for table in tables:
    table_name = table[0]
    count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(table_name, count)

docs 25565
sqlite_sequence 1
docs_fts 25565
docs_fts_data 10151
docs_fts_idx 5169
docs_fts_content 25565
docs_fts_docsize 25565
docs_fts_config 1


# Rag Helper Keyword Index

In [11]:
ingest.build_keyword_index(documents)



Keyword index saved to jobs.db


In [12]:
from openai import OpenAI
import rag_helper

# load the keyword search index we created in ingest.py

index = rag_helper.load_index()
client = OpenAI()

rag = rag_helper.RAGBase(index=index, llm_client=client)

In [13]:
results = rag.vector_search("Recommend Tel Aviv jobs for someone with SQL, Python and product analytics")

results[0]

{'Title': 'Junior Research Analyst',
 'Job_Description': '(Entry-Level | No Prior Experience Required) StoreNext is a leading technology company specializing in research and market data analysis within Israel’s Fast-Moving Consumer Goods (FMCG) industry. We provide data-driven insights to brand, marketing, and commercial leaders, directly influencing strategic decisions in one of the most dynamic and competitive markets We are looking for a motivated and analytical Junior Analyst to join our research team This is an excellent entry-level opportunity for recent graduates who are eager to begin a career in data analysis, market research, and business consulting — working with real market data from thousands of retail stores across Israel : Responsibilities Process and analyze large-scale point-of-sale (POS) data from thousands of retail locations Generate insights and identify key trends in the Israeli consumer goods market Conduct market reviews based on real sales data and consumer beh

In [14]:
results = rag.hybrid_search(
    "Recommend Tel Aviv jobs for someone with SQL, Python and product analytics"
)

results

[{'Title': 'Data Analyst',
  'Job_Description': "Papaya is one of the world's fastest-growing mobile game companies, providing entertainment for millions of users across multiple apps. The company is headquartered in the heart of Tel Aviv. We are looking for a sharp and passionate Business Analyst to join our top-notch analytics team. Using your technical and analytical skills, your mission will be to tell the story behind the numbers, understand how players interact with our games, and leverage our data to propose new ideas that you’ll then see implemented.Responsibilities: Use statistical and mathematical methods to recommend new ways to optimize and maximize player retention and game revenues.Analyze large, complex data sets to solve business questions and be a data-oriented users behavior expert.Build dashboards and reports to facilitate understanding of key business metrics and players' experience.Work closely with developers and product teams to execute new features and processes

In [15]:
answer = rag.rag(
    "Recommend Tel Aviv jobs for someone with SQL, Python and product analytics"
)

answer


'Here are the most relevant **Tel Aviv jobs** for someone with **SQL, Python, and product analytics** skills:\n\n1. **Business Analyst — PAPAYA**\n   - **City:** Tel Aviv-Yafo\n   - **Why it matches:** Strong fit for product analytics work. The role focuses on analyzing user behavior, optimizing game revenue and retention, and building dashboards.\n   - **Relevant skills:** SQL, Tableau/Looker, A/B testing, Python, statistics\n   - **Experience:** 4+ years\n   - **Link:** https://il.linkedin.com/jobs/view/business-analyst-at-papaya-4387814536\n\n2. **Data Analyst — PAPAYA**\n   - **City:** Tel Aviv-Yafo\n   - **Why it matches:** Very aligned with product analytics and data analysis in a gaming context. You’d analyze large datasets, understand player behavior, and help optimize retention and revenue.\n   - **Relevant skills:** SQL, Python, Tableau/Looker, A/B testing, statistics\n   - **Experience:** 3+ years listed, but description asks for 4+ years in practice\n   - **Link:** https://

In [16]:
rag.rag("Find good data analyst jobs for someone with 3 years of experience")

'Here are the best matches for someone with **3 years of experience** in a data analyst–type path:\n\n### 1) Intelligence Analyst Threat Hunter — Chainalysis\n- **Location:** Tel Aviv, Israel\n- **Experience:** **2+ years**\n- **Why it fits:** This is the strongest match for your experience level. It involves **SQL**, **Python**, data pipeline monitoring, research, and working with ambiguous data problems. It’s especially relevant if you’re interested in **fraud, compliance, threat intelligence, or crypto/Web3 data**.\n- **Good if you have:** SQL, Python, AI/LLM familiarity, analytical thinking, and interest in investigative data work.\n\n### 2) Junior Research Analyst — StoreNext\n- **Location:** Tel Aviv-Yafo / Raanana\n- **Experience:** **0+**\n- **Why it fits:** Even though it’s entry-level, someone with 3 years of experience would still qualify. The role is very aligned with **data analysis, market research, insights, Excel, and business recommendations**.\n- **Good if you want:**

In [17]:
rag.rag("Find jobs that are good for someone with strong communication and presentation skills")

'These jobs look especially good for someone with strong communication and presentation skills:\n\n1. **Junior Research Analyst — StoreNext**\n   - **Locations:** Raanana, Center District, Israel\n   - **Experience:** 0+\n   - **Why it fits:** This role explicitly includes **presentation skills** and working directly with clients. You’ll present insights and conclusions to decision-makers, so communication is a key part of the job.\n\n2. **Junior Research Analyst — StoreNext**\n   - **Location:** Tel Aviv-Yafo, Tel Aviv District, Israel\n   - **Experience:** 0+\n   - **Why it fits:** Same role, also explicitly requires **presentation skills** and involves presenting findings and recommendations to clients and business leaders.\n\n3. **Customer Success Associate — Ripples**\n   - **Location:** Petah Tikva, Center District, Israel\n   - **Experience:** 0+\n   - **Why it fits:** This role is heavily customer-facing. It requires **excellent English communication skills**, onboarding custom